# Module 03 -- FX Options (Garman-Kohlhagen)

**Author: Djellal Djouad** -- CrossVol Research | [crossvol.com](https://crossvol.com) | ORCID [0009-0002-4911-1118](https://orcid.org/0009-0002-4911-1118)

FX options are the forgotten corner of open-source quant finance. There are dozens
of equity option repos on GitHub, but almost nothing for FX that handles the
conventions correctly. This matters because FX has its own language: premium-adjusted
delta, forward delta, straddle quotes in vol terms, strike from delta... if you
port equity intuition directly to FX, you'll get the signs and sizes wrong.

I spent years on an FX derivatives desk and the number of times I saw people
confuse domestic and foreign rates, or forget the premium adjustment on delta,
was alarming. This notebook builds it from scratch so you understand every piece.

*License: MIT with Educational Use Clause -- see LICENSE. Not trading advice.*

**References:**
- *FX Options: From Theory to Practice* -- [Amazon](https://www.amazon.com/dp/B0H3VSV88X)
- Djouad (2025), *FX Derivatives Framework* -- [doi:10.5281/zenodo.20509708](https://doi.org/10.5281/zenodo.20509708)


In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt


## Garman-Kohlhagen: BSM Adapted for FX

The key difference from equity BSM: instead of a dividend yield, you have a
**foreign risk-free rate** ($r_f$). The domestic rate is $r_d$. The spot is
quoted as units of domestic per one unit of foreign (e.g., EURUSD = 1.08 means
1 EUR costs 1.08 USD).

$$C = S \cdot e^{-r_f T} N(d_1) - K \cdot e^{-r_d T} N(d_2)$$

$$d_1 = \frac{\ln(S/K) + (r_d - r_f + \frac{\sigma^2}{2})T}{\sigma\sqrt{T}}$$


In [ ]:
def gk_price(S, K, T, rd, rf, sigma, option_type='call'):
    """Garman-Kohlhagen pricing for FX options."""
    d1 = (np.log(S / K) + (rd - rf + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S * np.exp(-rf * T) * norm.cdf(d1) - K * np.exp(-rd * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-rd * T) * norm.cdf(-d2) - S * np.exp(-rf * T) * norm.cdf(-d1)
    return price

def gk_delta(S, K, T, rd, rf, sigma, option_type='call'):
    """Spot delta (not premium-adjusted)."""
    d1 = (np.log(S / K) + (rd - rf + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    if option_type == 'call':
        return np.exp(-rf * T) * norm.cdf(d1)
    else:
        return np.exp(-rf * T) * (norm.cdf(d1) - 1.0)

def gk_premium_adjusted_delta(S, K, T, rd, rf, sigma, option_type='call'):
    """
    Premium-adjusted delta -- the standard in FX markets.
    When you buy an FX option, you pay premium in one currency, which itself
    creates an FX exposure. The premium-adjusted delta accounts for this.
    """
    d1 = (np.log(S / K) + (rd - rf + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    _d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        spot_delta = np.exp(-rf * T) * norm.cdf(d1)
        premium_in_foreign = gk_price(S, K, T, rd, rf, sigma, 'call') / S
        return spot_delta - premium_in_foreign
    else:
        spot_delta = np.exp(-rf * T) * (norm.cdf(d1) - 1.0)
        premium_in_foreign = gk_price(S, K, T, rd, rf, sigma, 'put') / S
        return spot_delta + premium_in_foreign


## Forward Points & Forward Rate

The FX forward is determined by covered interest rate parity. No model needed --
it's pure arbitrage. If you can borrow in one currency and lend in another,
the forward adjusts so there's no free lunch.

$$F = S \cdot e^{(r_d - r_f) T}$$

Forward points = $F - S$, usually quoted in pips (4th decimal for most pairs).


In [ ]:
S = 1.0850  # EURUSD spot
rd = 0.045  # USD rate
rf = 0.035  # EUR rate

T_range = np.linspace(0.01, 2.0, 200)
forwards = S * np.exp((rd - rf) * T_range)
fwd_points = (forwards - S) * 10000  # in pips

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(T_range, forwards, color='steelblue', lw=2)
axes[0].axhline(S, color='grey', ls='--', lw=0.7, label=f'Spot = {S}')
axes[0].set_title('EURUSD Forward Curve')
axes[0].set_xlabel('Tenor (years)')
axes[0].set_ylabel('Forward Rate')
axes[0].legend()

axes[1].plot(T_range, fwd_points, color='firebrick', lw=2)
axes[1].axhline(0, color='grey', ls='--', lw=0.7)
axes[1].set_title('Forward Points (pips)')
axes[1].set_xlabel('Tenor (years)')
axes[1].set_ylabel('Pips')

plt.tight_layout()
plt.show()


When $r_d > r_f$ (USD rates above EUR rates), the forward is above spot -- EURUSD
trades at a premium forward. This means the market "expects" (in a risk-neutral
sense) EUR to appreciate, because holding USD earns more carry and the forward
must offset that advantage.


## Spot Delta vs. Premium-Adjusted Delta

This trips up almost everyone coming from equity land. In FX, when you buy a call
on EURUSD, you pay the premium in USD (domestic). But that premium payment itself
is an FX transaction -- you're selling USD to buy the option. So your true delta
exposure is the "raw" delta minus the premium you paid expressed in foreign terms.

For major pairs (EURUSD, GBPUSD), the market quotes premium-adjusted delta.
For EM pairs, conventions vary. Always check. I've seen blown hedges from
getting this wrong on a USDTRY book.


In [ ]:
K_range = np.linspace(1.02, 1.15, 200)
T = 0.25  # 3 months
sigma = 0.08  # 8% vol, typical for EURUSD

spot_d = gk_delta(S, K_range, T, rd, rf, sigma, 'call')
pa_d = gk_premium_adjusted_delta(S, K_range, T, rd, rf, sigma, 'call')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, spot_d, label='Spot Delta', lw=2)
ax.plot(K_range, pa_d, label='Premium-Adjusted Delta', lw=2, ls='--')
ax.axvline(S, color='grey', ls=':', lw=0.7, label=f'Spot = {S}')
ax.set_xlabel('Strike')
ax.set_ylabel('Delta')
ax.set_title('Call Delta: Spot vs. Premium-Adjusted (EURUSD, 3M)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Notice the premium-adjusted delta is always lower than the spot delta for calls.
That gap widens for ITM options where the premium is larger. For deep OTM options,
the two converge because the premium is negligible.


## Option Prices Across Strikes


In [ ]:
call_prices = gk_price(S, K_range, T, rd, rf, sigma, 'call')
put_prices = gk_price(S, K_range, T, rd, rf, sigma, 'put')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, call_prices * 10000, label='Call (pips)', color='steelblue', lw=2)
ax.plot(K_range, put_prices * 10000, label='Put (pips)', color='firebrick', lw=2)
ax.axvline(S, color='grey', ls=':', lw=0.7)
ax.set_xlabel('Strike')
ax.set_ylabel('Premium (pips)')
ax.set_title('GK Option Prices -- EURUSD 3M, 8% vol')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Comparing GK to Equity BSM

Quick sanity check: if $r_f = 0$ (no foreign rate), Garman-Kohlhagen collapses
to standard BSM with no dividends. Let's verify.


In [ ]:
from scipy.stats import norm as norm_dist

def bsm_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm_dist.cdf(d1) - K * np.exp(-r * T) * norm_dist.cdf(d2)

# With rf=0, GK should match BSM exactly
test_S, test_K, test_T, test_r, test_sigma = 100, 100, 0.5, 0.05, 0.20

gk_val = gk_price(test_S, test_K, test_T, test_r, 0.0, test_sigma, 'call')
bsm_val = bsm_call(test_S, test_K, test_T, test_r, test_sigma)

print(f"GK (rf=0):  {gk_val:.6f}")
print(f"BSM:        {bsm_val:.6f}")
print(f"Difference: {abs(gk_val - bsm_val):.2e}")


## Desk Notes

A few things I wish someone had told me before I started trading FX options:

1. **Volatility is quoted, not price.** The FX option market trades in vol. You
   agree on a vol with your counterparty, then both sides compute the premium
   independently. If your models disagree on the third decimal, you have a problem.

2. **The smile is quoted in delta space**, not strike space. A "25-delta put" is a
   put whose delta is -0.25. The strike is implied from the vol and the delta.
   This is backwards from equity thinking.

3. **Tenors are standardized** (1W, 1M, 2M, 3M, 6M, 1Y, 2Y) and the expiry/delivery
   dates follow FX settlement conventions (T+2 spot, modified following).

4. **Strangles and risk reversals** are the building blocks, not individual puts
   and calls. The 25D RR tells you skew. The 25D BF (butterfly) tells you kurtosis.

---

**Next:** [Module 04 -- Implied Volatility & Smile](04_implied_vol.py)

*Djellal Djouad -- CrossVol Research -- 2026*
